# Phase 4 - GSM-Symbolic / GSM-Plus memorisation test

Runs the selected models on original GSM8K questions and on re-instantiated / perturbed versions of the same questions. The accuracy drop is the ground-truth contamination signal used to validate the v1-v2 inflation proxy.

**Before running**
1. *Runtime -> Change runtime type -> T4 GPU*.
2. Add your Hugging Face read token in the key icon (**Secrets**) on the left, named `HF_TOKEN`, and switch on notebook access. Never paste it into a cell.
3. *Runtime -> Run all*. Predictions are saved to Google Drive after each model, so if Colab disconnects, just run all again: finished models are skipped.

**Kaggle (2x T4, for the 7-8B tier):** use the same cells but skip the Drive cell, set `OUT = '/kaggle/working/phase4'`, `TIERS = 'kaggle'`, `TP = 2`, and add `HF_TOKEN` under *Add-ons -> Secrets*.

In [ ]:
TIERS = 'local,colab'   # models up to 4B fit a T4 in fp16
TP = 1                  # GPUs for vLLM tensor parallel (Kaggle 2x T4: 2)
OUT = '/content/drive/MyDrive/ada_phase4'
REPO = 'https://github.com/minkysponge-wq/Ada-Mini-project.git'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.exists('/content/ada'):
    !git clone -q {REPO} /content/ada
%cd /content/ada
!git pull -q
!pip install -q vllm pandas networkx scipy matplotlib
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# quick smoke test: one small model, 3 seed questions per dataset
!python phase4_eval.py --backend vllm --tp {TP} --out {OUT}_smoke --limit 3 --only Qwen/Qwen2-0.5B

In [ ]:
# full run (resumes: finished models are skipped)
!python phase4_eval.py --models out_phase1/phase4/models.csv --tiers {TIERS} --backend vllm --tp {TP} --out {OUT}

In [ ]:
# analysis + zip of everything to download into ada_contam/out_phase1/phase4/
!python phase4_analyze.py --models out_phase1/phase4/models.csv --preds {OUT}/preds --out {OUT}
!cd {OUT}/.. && zip -qr ada_phase4_results.zip $(basename {OUT})
print('Download from Drive:', os.path.dirname(OUT) + '/ada_phase4_results.zip')